[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/01_Linear_Regression_Example/Linear_Regression_Example_Apply.ipynb)

# 1.1 Linear Regression in ONNX — Hands-On Practice

Build, validate, visualize, and **run** linear regression ONNX models end-to-end.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Setup & Imports](#section-1) | Install and import dependencies |
| 2 | [Exercise 1: Build & Run Y = XA + B](#section-2) | Core graph construction and inference |
| 3 | [Exercise 2: Visualize the Graph](#section-3) | Matplotlib-based graph rendering |
| 4 | [Exercise 3: Modify to Y = XA − B](#section-4) | Swap operators |
| 5 | [Exercise 4: Three-Operator Pipeline](#section-5) | Y = Abs(XA + B) |
| 6 | [Exercise 5: Multi-Output Regression](#section-6) | Multiple target variables |
| 7 | [Exercise 6: Batch Size Experiments](#section-7) | Dynamic shapes in action |
| 8 | [Exercise 7: Performance Measurement](#section-8) | Benchmarking inference speed |
| 9 | [Challenge: Build a 2-Layer Network](#section-9) | Compose multiple linear transforms |

<a id='section-1'></a>
## Section 1: Setup & Imports

In [ ]:
# Uncomment for Colab:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time

from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info)
from onnx.checker import check_model
import onnxruntime as ort

print(f'ONNX Runtime version: {ort.__version__}')
print('Setup complete!')

<a id='section-2'></a>
## Section 2: Exercise 1 — Build & Run $Y = XA + B$

### Task

Construct an ONNX graph for the linear regression equation $Y = XA + B$ using two nodes (`MatMul` → `Add`), then run inference with concrete values.

### Procedure

1. Declare three inputs ($X$, $A$, $B$) and one output ($Y$) as `ValueInfoProto`
2. Create two `NodeProto` objects: `MatMul` and `Add`
3. Wire them via the intermediate name `'XA'`
4. Assemble into a graph, wrap into a model, and validate
5. Run with ONNX Runtime and verify against NumPy

In [ ]:
# Step 1: Declare typed tensor interfaces
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

# Step 2: Create operator nodes
node1 = make_node('MatMul', ['X', 'A'], ['XA'])
node2 = make_node('Add', ['XA', 'B'], ['Y'])

# Step 3: Assemble and validate
graph = make_graph([node1, node2], 'lr', [X, A, B], [Y])
onnx_model = make_model(graph)
check_model(onnx_model)

# Step 4: Run inference
sess = ort.InferenceSession(
    onnx_model.SerializeToString(),
    providers=['CPUExecutionProvider'])

x = np.array([[1, 2], [3, 4], [5, 6], [7, 8]], dtype=np.float32)
a = np.array([[0.5], [-0.3]], dtype=np.float32)
b = np.array([[1.0]], dtype=np.float32)

result = sess.run(None, {'X': x, 'A': a, 'B': b})

print('ONNX result:  ', result[0].flatten())
print('NumPy check:  ', (x @ a + b).flatten())
print('Match:        ', np.allclose(result[0], x @ a + b))

<a id='section-3'></a>
## Section 3: Exercise 2 — Visualize the Graph

### Task

Write a **reusable function** that takes any ONNX model and renders its computation graph using matplotlib. This function should:
- Show input nodes (blue), operator nodes (yellow), and output nodes (green)
- Draw edges with arrows between connected nodes
- Label each node with its name and type

In [ ]:
def visualize_onnx_graph(model, title='ONNX Computation Graph', figsize=(10, 8)):
    """Render an ONNX model's graph as a matplotlib figure."""
    graph = model.graph
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)

    styles = {
        'input':  dict(boxstyle='round,pad=0.5', fc='#AED6F1', ec='#2471A3', lw=2),
        'op':     dict(boxstyle='round,pad=0.5', fc='#F9E79F', ec='#B7950B', lw=2),
        'output': dict(boxstyle='round,pad=0.5', fc='#A9DFBF', ec='#1E8449', lw=2),
    }
    positions = {}
    n_inputs = len(graph.input)
    n_nodes = len(graph.node)
    n_outputs = len(graph.output)

    total_width = max(n_inputs, n_nodes, n_outputs) * 2.5
    y_levels = [6, 4, 2, 0]  # inputs, operator-rows, ..., outputs

    # Place inputs
    for i, inp in enumerate(graph.input):
        x_pos = (i + 0.5) * total_width / n_inputs
        positions[inp.name] = (x_pos, y_levels[0])
        ax.text(x_pos, y_levels[0], f'{inp.name}\n(input)', ha='center',
                va='center', fontsize=10, fontweight='bold', bbox=styles['input'])

    # Place operator nodes
    for i, node in enumerate(graph.node):
        x_pos = total_width / 2
        y_pos = y_levels[1] - i * 1.8
        node_id = node.output[0] if node.output else f'node_{i}'
        positions[node_id] = (x_pos, y_pos)
        ax.text(x_pos, y_pos, f'{node.op_type}\n→ {node_id}', ha='center',
                va='center', fontsize=10, fontweight='bold', bbox=styles['op'])

        for in_name in node.input:
            if in_name in positions:
                sx, sy = positions[in_name]
                ax.annotate('', xy=(x_pos, y_pos + 0.5), xytext=(sx, sy - 0.5),
                           arrowprops=dict(arrowstyle='->', lw=1.5, color='#555'))

    # Place outputs
    last_node_y = y_levels[1] - (n_nodes - 1) * 1.8
    for i, out in enumerate(graph.output):
        x_pos = total_width / 2
        y_pos = last_node_y - 2
        ax.text(x_pos, y_pos, f'{out.name}\n(output)', ha='center',
                va='center', fontsize=10, fontweight='bold', bbox=styles['output'])
        if out.name in positions:
            sx, sy = positions[out.name]
            ax.annotate('', xy=(x_pos, y_pos + 0.5), xytext=(sx, sy - 0.5),
                       arrowprops=dict(arrowstyle='->', lw=1.5, color='#555'))

    ax.set_xlim(-0.5, total_width + 0.5)
    ax.set_ylim(last_node_y - 3.5, y_levels[0] + 1.5)

    legend_els = [
        mpatches.Patch(fc='#AED6F1', ec='#2471A3', label='Input'),
        mpatches.Patch(fc='#F9E79F', ec='#B7950B', label='Operator'),
        mpatches.Patch(fc='#A9DFBF', ec='#1E8449', label='Output'),
    ]
    ax.legend(handles=legend_els, loc='upper right')
    plt.tight_layout()
    plt.show()

visualize_onnx_graph(onnx_model, 'Exercise 2: Linear Regression Graph')

<a id='section-4'></a>
## Section 4: Exercise 3 — Modify to $Y = XA - B$

### Task

Change the model to compute subtraction instead of addition. Simply replace the `'Add'` operator with `'Sub'`.

### Mathematical Formulation

$$Y = XA - B \quad \Longrightarrow \quad Y_i = \sum_d X_{id} \cdot A_d - B$$

This exercise demonstrates that swapping operators is trivial once you understand the graph structure — the wiring (edge names) stays the same.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

node1 = make_node('MatMul', ['X', 'A'], ['XA'])
node2 = make_node('Sub', ['XA', 'B'], ['Y'])  # Sub instead of Add

graph = make_graph([node1, node2], 'lr_sub', [X, A, B], [Y])
model_sub = make_model(graph)
check_model(model_sub)

sess_sub = ort.InferenceSession(
    model_sub.SerializeToString(),
    providers=['CPUExecutionProvider'])

x = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
a = np.array([[0.5], [-0.3]], dtype=np.float32)
b = np.array([[1.0]], dtype=np.float32)

result_sub = sess_sub.run(None, {'X': x, 'A': a, 'B': b})
expected = x @ a - b

print('Y = XA - B')
print(f'  ONNX result:  {result_sub[0].flatten()}')
print(f'  NumPy check:  {expected.flatten()}')
print(f'  Match: {np.allclose(result_sub[0], expected)}')

<a id='section-5'></a>
## Section 5: Exercise 4 — Three-Operator Pipeline $Y = |XA + B|$

### Task

Extend the graph to include a third node: an `Abs` (absolute value) operator applied to the result of `Add`. This creates a three-node chain:

```
X, A ──► MatMul ──► XA ──► Add ──► XAB ──► Abs ──► Y
                           ▲
                    B ─────┘
```

### Mathematical Formulation

$$Y = |XA + B| \quad \Longrightarrow \quad Y_{ij} = \left| \sum_{d=1}^{D} X_{id} \cdot A_{dj} + B_j \right|$$

Notice that the intermediate tensor between `Add` and `Abs` needs a new name (`'XAB'` instead of `'Y'`), because `'Y'` is now produced by the `Abs` node.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

node1 = make_node('MatMul', ['X', 'A'], ['XA'])
node2 = make_node('Add', ['XA', 'B'], ['XAB'])   # intermediate name
node3 = make_node('Abs', ['XAB'], ['Y'])          # final output

graph = make_graph([node1, node2, node3], 'lr_abs', [X, A, B], [Y])
model_abs = make_model(graph)
check_model(model_abs)

sess_abs = ort.InferenceSession(
    model_abs.SerializeToString(),
    providers=['CPUExecutionProvider'])

x = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
a = np.array([[0.5], [-0.3]], dtype=np.float32)
b = np.array([[-2.0]], dtype=np.float32)  # large negative bias to create negatives

result_abs = sess_abs.run(None, {'X': x, 'A': a, 'B': b})
expected_abs = np.abs(x @ a + b)

print('Y = |XA + B|')
print(f'  Before Abs (XA+B): {(x @ a + b).flatten()}')
print(f'  After Abs:         {result_abs[0].flatten()}')
print(f'  NumPy check:       {expected_abs.flatten()}')
print(f'  Match: {np.allclose(result_abs[0], expected_abs)}')

In [ ]:
visualize_onnx_graph(model_abs, 'Exercise 4: Y = |XA + B| (3 nodes)')

<a id='section-6'></a>
## Section 6: Exercise 5 — Multi-Output Regression

### Task

Build a model that predicts **two separate outputs** from the same input: $Y_1 = XA_1 + B_1$ and $Y_2 = XA_2 + B_2$. This creates a graph with two parallel branches.

```
         ┌──► MatMul(X,A1) ──► Add(·,B1) ──► Y1
    X ───┤
         └──► MatMul(X,A2) ──► Add(·,B2) ──► Y2
```

This pattern is common in multi-task learning where a shared backbone feeds into separate prediction heads.

In [ ]:
X  = make_tensor_value_info('X',  TensorProto.FLOAT, [None, 4])
A1 = make_tensor_value_info('A1', TensorProto.FLOAT, [4, 1])
B1 = make_tensor_value_info('B1', TensorProto.FLOAT, [1, 1])
A2 = make_tensor_value_info('A2', TensorProto.FLOAT, [4, 1])
B2 = make_tensor_value_info('B2', TensorProto.FLOAT, [1, 1])
Y1 = make_tensor_value_info('Y1', TensorProto.FLOAT, [None, 1])
Y2 = make_tensor_value_info('Y2', TensorProto.FLOAT, [None, 1])

# Branch 1: Y1 = X @ A1 + B1
n1 = make_node('MatMul', ['X', 'A1'], ['XA1'])
n2 = make_node('Add',    ['XA1', 'B1'], ['Y1'])

# Branch 2: Y2 = X @ A2 + B2
n3 = make_node('MatMul', ['X', 'A2'], ['XA2'])
n4 = make_node('Add',    ['XA2', 'B2'], ['Y2'])

graph = make_graph(
    [n1, n2, n3, n4], 'multi_output',
    [X, A1, B1, A2, B2], [Y1, Y2])
model_multi = make_model(graph)
check_model(model_multi)

sess_multi = ort.InferenceSession(
    model_multi.SerializeToString(),
    providers=['CPUExecutionProvider'])

x  = np.random.randn(5, 4).astype(np.float32)
a1 = np.random.randn(4, 1).astype(np.float32)
b1 = np.array([[0.5]], dtype=np.float32)
a2 = np.random.randn(4, 1).astype(np.float32)
b2 = np.array([[-0.5]], dtype=np.float32)

y1_onnx, y2_onnx = sess_multi.run(None, {
    'X': x, 'A1': a1, 'B1': b1, 'A2': a2, 'B2': b2})

print('Multi-output model (5 samples, 4 features):')
print(f'  Y1 shape: {y1_onnx.shape}, match: {np.allclose(y1_onnx, x @ a1 + b1)}')
print(f'  Y2 shape: {y2_onnx.shape}, match: {np.allclose(y2_onnx, x @ a2 + b2)}')
print(f'  Model has {len(model_multi.graph.node)} nodes, '
      f'{len(model_multi.graph.input)} inputs, '
      f'{len(model_multi.graph.output)} outputs')

<a id='section-7'></a>
## Section 7: Exercise 6 — Batch Size Experiments

### Task

Using the original $Y = XA + B$ model with dynamic shapes, run inference with various batch sizes and plot the output shapes and latencies.

In [ ]:
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
latencies = []

a_fixed = np.array([[0.5], [-0.3]], dtype=np.float32)
b_fixed = np.array([[1.0]], dtype=np.float32)

for bs in batch_sizes:
    x_test = np.random.randn(bs, 2).astype(np.float32)
    feeds = {'X': x_test, 'A': a_fixed, 'B': b_fixed}

    # Warm-up
    sess.run(None, feeds)

    # Measure
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        sess.run(None, feeds)
        times.append((time.perf_counter() - t0) * 1e6)  # microseconds

    avg_us = np.mean(times)
    latencies.append(avg_us)
    print(f'  batch={bs:5d}  latency={avg_us:8.1f} µs')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, latencies, 'o-', color='#2E86C1', linewidth=2, markersize=7)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Latency (µs)', fontsize=12)
ax.set_title('Inference Latency vs Batch Size (Y = XA + B)', fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Exercise 7 — Performance Measurement

### Task

Compare ONNX Runtime inference against NumPy for varying matrix sizes. Measure and plot the throughput (samples per second) for both backends.

In [ ]:
feature_sizes = [2, 8, 32, 128, 512]
n_samples = 1000
n_runs = 100

ort_throughputs = []
np_throughputs = []

for d in feature_sizes:
    # Build model for this feature size
    _X = make_tensor_value_info('X', TensorProto.FLOAT, [None, d])
    _A = make_tensor_value_info('A', TensorProto.FLOAT, [d, 1])
    _B = make_tensor_value_info('B', TensorProto.FLOAT, [1, 1])
    _Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 1])

    _g = make_graph(
        [make_node('MatMul', ['X', 'A'], ['XA']),
         make_node('Add', ['XA', 'B'], ['Y'])],
        'bench', [_X, _A, _B], [_Y])
    _m = make_model(_g)

    _sess = ort.InferenceSession(
        _m.SerializeToString(),
        providers=['CPUExecutionProvider'])

    x_data = np.random.randn(n_samples, d).astype(np.float32)
    a_data = np.random.randn(d, 1).astype(np.float32)
    b_data = np.array([[0.1]], dtype=np.float32)
    feeds = {'X': x_data, 'A': a_data, 'B': b_data}

    # Warm up
    _sess.run(None, feeds)
    _ = x_data @ a_data + b_data

    # ORT timing
    t0 = time.perf_counter()
    for _ in range(n_runs):
        _sess.run(None, feeds)
    ort_time = time.perf_counter() - t0

    # NumPy timing
    t0 = time.perf_counter()
    for _ in range(n_runs):
        _ = x_data @ a_data + b_data
    np_time = time.perf_counter() - t0

    ort_throughputs.append(n_samples * n_runs / ort_time)
    np_throughputs.append(n_samples * n_runs / np_time)

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(feature_sizes))
width = 0.35
ax.bar(x_pos - width/2, [t/1e6 for t in ort_throughputs], width,
       label='ONNX Runtime', color='#2E86C1', alpha=0.8)
ax.bar(x_pos + width/2, [t/1e6 for t in np_throughputs], width,
       label='NumPy', color='#E74C3C', alpha=0.8)
ax.set_xlabel('Feature Dimension (D)', fontsize=12)
ax.set_ylabel('Throughput (M samples/sec)', fontsize=12)
ax.set_title('ONNX Runtime vs NumPy: Linear Regression Throughput', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_sizes)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

<a id='section-9'></a>
## Section 9: Challenge — Build a 2-Layer Neural Network

### Task

Build a model that computes a two-layer forward pass:

$$H = \text{Relu}(X W_1 + B_1) \qquad \text{(hidden layer)}$$
$$Y = H W_2 + B_2 \qquad \text{(output layer)}$$

This requires **five** nodes: `MatMul` → `Add` → `Relu` → `MatMul` → `Add`.

```
X ──► MatMul(X,W1) ──► Add(·,B1) ──► Relu ──► MatMul(·,W2) ──► Add(·,B2) ──► Y
```

In [ ]:
X  = make_tensor_value_info('X',  TensorProto.FLOAT, [None, 4])
W1 = make_tensor_value_info('W1', TensorProto.FLOAT, [4, 8])
B1 = make_tensor_value_info('B1', TensorProto.FLOAT, [1, 8])
W2 = make_tensor_value_info('W2', TensorProto.FLOAT, [8, 2])
B2 = make_tensor_value_info('B2', TensorProto.FLOAT, [1, 2])
Y  = make_tensor_value_info('Y',  TensorProto.FLOAT, [None, 2])

# Layer 1: H = Relu(X @ W1 + B1)
n1 = make_node('MatMul', ['X', 'W1'], ['XW1'])
n2 = make_node('Add',    ['XW1', 'B1'], ['pre_relu'])
n3 = make_node('Relu',   ['pre_relu'], ['H'])

# Layer 2: Y = H @ W2 + B2
n4 = make_node('MatMul', ['H', 'W2'], ['HW2'])
n5 = make_node('Add',    ['HW2', 'B2'], ['Y'])

graph = make_graph(
    [n1, n2, n3, n4, n5], 'two_layer_net',
    [X, W1, B1, W2, B2], [Y])
model_2layer = make_model(graph)
check_model(model_2layer)

sess_2l = ort.InferenceSession(
    model_2layer.SerializeToString(),
    providers=['CPUExecutionProvider'])

# Test data
x_test  = np.random.randn(10, 4).astype(np.float32)
w1_test = np.random.randn(4, 8).astype(np.float32) * 0.1
b1_test = np.zeros((1, 8), dtype=np.float32)
w2_test = np.random.randn(8, 2).astype(np.float32) * 0.1
b2_test = np.zeros((1, 2), dtype=np.float32)

y_onnx = sess_2l.run(None, {
    'X': x_test, 'W1': w1_test, 'B1': b1_test,
    'W2': w2_test, 'B2': b2_test})[0]

# NumPy reference
h_np = np.maximum(0, x_test @ w1_test + b1_test)  # Relu
y_np = h_np @ w2_test + b2_test

print(f'Two-layer network:')
print(f'  Input shape:  {x_test.shape}')
print(f'  Hidden shape: {h_np.shape}')
print(f'  Output shape: {y_onnx.shape}')
print(f'  Match: {np.allclose(y_onnx, y_np, atol=1e-6)}')
print(f'  Nodes: {len(model_2layer.graph.node)}')
print(f'  First 3 predictions (ONNX):  {y_onnx[:3].tolist()}')
print(f'  First 3 predictions (NumPy): {y_np[:3].tolist()}')

In [ ]:
visualize_onnx_graph(model_2layer, 'Challenge: 2-Layer Neural Network')

---

## Summary

In this notebook you practiced:

| Exercise | Skill | Nodes |
|----------|-------|-------|
| 1 | Build & run a basic graph | 2 |
| 2 | Visualize any ONNX graph | — |
| 3 | Swap operators (Add → Sub) | 2 |
| 4 | Chain three operators | 3 |
| 5 | Multi-output models | 4 |
| 6 | Dynamic batch sizes | 2 |
| 7 | Performance benchmarking | 2 |
| Challenge | Multi-layer network | 5 |

**Next:** [Serialization](../02_Serialization/) — Learn to save and load ONNX models.